In [0]:
# Databricks Community Edition
df = spark.table("samples.nyctaxi.trips")

# Local PySpark alternative (Requires Hadoop AWS packages):
# df = spark.read.parquet("s3a://nyc-tlc/trip data/yellow_tripdata_2024-01.parquet")

df.show(5)

In [0]:
# View the dataframe schema and datatypes
df.printSchema()

# Generate summary statistics for key numerical columns
df.describe(["fare_amount", "trip_distance"]).show()

# Count the total number of records
print(f"Total trips processed: {df.count():,}")

In [0]:
df.createOrReplaceTempView("nyc_taxi")

# Find the top 10 pickup ZIP codes by total revenue, ignoring zero-distance trips
agg_df = spark.sql("""
    SELECT 
        pickup_zip,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(SUM(fare_amount), 2) AS total_revenue
    FROM nyc_taxi
    WHERE trip_distance > 0 AND fare_amount > 0
    GROUP BY pickup_zip
    ORDER BY total_revenue DESC
    LIMIT 10
""")

agg_df.show()

In [0]:
from pyspark.sql.functions import to_date

# Create a clean date column to use as our partition key
df_prepared = df.withColumn("pickup_date", to_date("tpep_pickup_datetime"))

# Write Unpartitioned (Default flat Parquet)
df_prepared.write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/raw_data/nyc_taxi/unpartitioned")

# Write Partitioned by Date (Creates hierarchical folders like pickup_date=2016-02-14/)
df_prepared.write.mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet("/Volumes/workspace/default/raw_data/nyc_taxi/partitioned")

In [0]:
import time

def benchmark_query(path, query_date):
    test_df = spark.read.parquet(path)
    test_df.createOrReplaceTempView("benchmark_view")
    
    start_time = time.time()
    
    # .collect() triggers the execution DAG
    spark.sql(f"""
        SELECT COUNT(*), AVG(fare_amount) 
        FROM benchmark_view 
        WHERE pickup_date = '{query_date}'
    """).collect()
    
    return time.time() - start_time

# Pick a date that exists in your specific DataFrame sample
test_date = "2016-02-14"

time_unpart = benchmark_query("/Volumes/workspace/default/raw_data/nyc_taxi/unpartitioned", test_date)
time_part = benchmark_query("/Volumes/workspace/default/raw_data/nyc_taxi/partitioned", test_date)

print(f"Full Table Scan (Unpartitioned): {time_unpart:.2f} seconds")
print(f"Partition Pruning (Partitioned): {time_part:.2f} seconds")
print(f"Speedup: {time_unpart / time_part:.1f}x faster")

In [0]:
# 1. Define the schema based on the batch DataFrame
taxi_schema = df_prepared.schema

# 2. Read as a stream (processing 1 file per trigger to simulate live incoming drops)
streaming_df = spark.readStream \
    .schema(taxi_schema) \
    .option("maxFilesPerTrigger", 1) \
    .parquet("/Volumes/workspace/default/raw_data/nyc_taxi/unpartitioned")

# 3. Define the live streaming aggregation
streaming_agg = streaming_df \
    .groupBy("pickup_zip") \
    .count()

# 4. Start the stream and output to an in-memory table
query = streaming_agg.writeStream \
    .format("memory") \
    .queryName("live_taxi_counts") \
    .outputMode("complete") \
    .option("checkpointLocation", "/Volumes/workspace/default/raw_data/nyc_taxi/checkpoints/live_taxi_counts") \
    .trigger(availableNow=True) \
    .start()
    
# Wait ~10 seconds for the stream to process a few files, then query the live table in a new cell:
# spark.sql("SELECT * FROM live_taxi_counts ORDER BY count DESC LIMIT 5").show()

# Stop the stream when finished:
# query.stop()

In [0]:
# Cell 1: Import and Read Raw Data
from pyspark.sql.functions import col, to_date

# Assuming a cluster or local setup pointing to raw CSV or Delta drops
# (s3a:// access is not available here — use the equivalent UC sample table)
raw_df = spark.table("samples.nyctaxi.trips")

# Cell 2: Clean and Create Partition Key
# Filter early (predicate pushdown) to remove negative fares and 0-distance errors
clean_df = raw_df.filter((col("trip_distance") > 0) & (col("fare_amount") > 0)) \
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

# Cell 3: Write Partitioned Parquet
# Partitioning by date logically splits the physical files on disk 
# (e.g., /pickup_date=2024-01-01/), enabling massive speedups via partition pruning.
clean_df.write.mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet("/Volumes/workspace/default/raw_data/nyc_taxi/clean_partitioned")

In [0]:
# Cell 1: Read and Register Temp View
# Spark will automatically recognize the 'pickup_date' partition folders
# (s3a:// access is not available here — read the partitioned volume data instead)
df = spark.read.parquet("/Volumes/workspace/default/raw_data/nyc_taxi/clean_partitioned")
df.createOrReplaceTempView("taxi_trips")

# Cell 2: Spark SQL Aggregation
# Querying a specific date automatically prunes the rest of the dataset from the scan
# (data spans 2016-01-01 to 2016-02-29; available columns differ from the raw S3 CSVs)
agg_df = spark.sql("""
    SELECT 
        pickup_date,
        pickup_zip,
        COUNT(1) as total_rides,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(SUM(trip_distance), 2) as total_distance
    FROM taxi_trips
    WHERE pickup_date >= '2016-02-01'
    GROUP BY pickup_date, pickup_zip
    ORDER BY pickup_date DESC
""")

# Cell 3: Action
agg_df.show(5)

In [0]:
# Cell 1: Define Schema and Read Stream
# Structured streaming requires a strict schema for file sources
# (s3a:// access is not available here — use the volume equivalents)
taxi_schema = spark.read.parquet("/Volumes/workspace/default/raw_data/nyc_taxi/clean_partitioned").schema

stream_df = spark.readStream \
    .schema(taxi_schema) \
    .option("maxFilesPerTrigger", 1) \
    .parquet("/Volumes/workspace/default/raw_data/nyc_taxi/unpartitioned")

# Cell 2: Define Continuous Aggregation
live_agg = stream_df.groupBy("pickup_zip").count()

# Cell 3: Start Stream Output (In-Memory Table for Notebook Viewing)
query = live_agg.writeStream \
    .format("memory") \
    .queryName("live_taxi_counts") \
    .outputMode("complete") \
    .option("checkpointLocation", "/Volumes/workspace/default/raw_data/nyc_taxi/checkpoints/live_taxi_counts_v2") \
    .trigger(availableNow=True) \
    .start()
    
# Cell 4: Query the live view in real-time
# spark.sql("SELECT * FROM live_taxi_counts ORDER BY count DESC LIMIT 5").show()